In [1]:
import matplotlib
matplotlib.use('Agg')           # 强制用 Agg，绕过 matplotlib_inline 的 bug
import matplotlib.pyplot as plt
import numpy as np

# 测试图能不能显示
fig, ax = plt.subplots()
ax.plot([1, 2, 3], [4, 2, 5])
ax.set_title("Test Plot")
plt.tight_layout()

from IPython.display import display as ipy_display
ipy_display(fig)
plt.close(fig)
print("Plot OK")

<Figure size 640x480 with 1 Axes>

Plot OK


In [ ]:
import sys
import os
import matplotlib
matplotlib.use('Agg')           # 必须在 import pyplot 之前
import matplotlib.pyplot as plt
from IPython.display import display as ipy_display

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import warnings

warnings.filterwarnings('ignore')

current_dir = Path.cwd()
models_dir = current_dir / 'Models'
if str(models_dir) not in sys.path:
    sys.path.append(str(models_dir))

from Models.base_model import TARGET_COL, EXCEEDANCE_THRESHOLD, WILDFIRE_COLUMNS
from Models.GAM_GRACH import GAMGARCHModel
from Models.lightboost import LightGBMModel
from Models.catboost_model import CatBoostModel
from Models.LSTM_AE_model import AutoencoderLSTMModel
from Models.Transformer_model import VanillaTransformerModel

# ─────────────────────────────────────────────────────────────
# DATASET CONFIG — 改這裡切換資料集
#   "original" : CS2_model_input.csv (原始, seq_len=48)
#   "pchip"    : CS2_pchip30min.csv  (30分鐘插值, seq_len=144)
#   "cvae"     : CS2_cvae_aug.npz    (CVAE 增廣訓練, seq_len=72)
#   "timegan"  : CS2_timegan_aug.npz (TimeGAN 增廣訓練, seq_len=72)
# ─────────────────────────────────────────────────────────────
DATASET_MODE = "pchip"

PROJECT_DIRECTORY = Path.cwd().parents[0]

if DATASET_MODE == "pchip":
    dataset_file_path = PROJECT_DIRECTORY / "Dataset" / "Outputs" / "CS2_pchip30min.csv"
    TRANSFORMER_SEQ_LEN = 144
    aug_data = None
elif DATASET_MODE in ("cvae", "timegan"):
    dataset_file_path = PROJECT_DIRECTORY / "Dataset" / "Outputs" / "CS2_model_input.csv"
    TRANSFORMER_SEQ_LEN = 48
    npz_path = PROJECT_DIRECTORY / "Dataset" / "Outputs" / f"CS2_{DATASET_MODE}_aug.npz"
    aug_data = np.load(str(npz_path))
    print(f"Loaded augmented data: {npz_path.name}")
else:
    dataset_file_path = PROJECT_DIRECTORY / "Dataset" / "Outputs" / "CS2_model_input.csv"
    TRANSFORMER_SEQ_LEN = 48
    aug_data = None

raw_input_dataframe = pd.read_csv(dataset_file_path)
raw_input_dataframe['Datetime_UTC'] = pd.to_datetime(raw_input_dataframe['Datetime_UTC'])
available_regions_list = raw_input_dataframe['Zone'].unique()
all_results = []

print(f"--- Running Unified Pipeline  [Mode={DATASET_MODE}]  Horizon=3 ---")

for region_name in available_regions_list:
    print(f"\nProcessing Region: {region_name} ...")
    regional_df = raw_input_dataframe[raw_input_dataframe['Zone'] == region_name].copy()
    regional_df = regional_df.set_index('Datetime_UTC').sort_index()
    if regional_df.index.tz is not None:
        regional_df.index = regional_df.index.tz_localize(None)

    train_data = regional_df[regional_df.index.year <= 2023].copy()
    val_data   = regional_df[regional_df.index.year == 2024].copy()
    test_data  = regional_df[regional_df.index.year == 2025].copy()

    if len(train_data) < 200:
        continue

    models_to_run = {
        # "1_GAM_Baseline":       GAMGARCHModel(candidate_features=[]),
        # "3_LightGBM_Wildfire":  LightGBMModel(candidate_features=WILDFIRE_COLUMNS),
        # "4_CatBoost_Wildfire":  CatBoostModel(candidate_features=WILDFIRE_COLUMNS),
        # "9_LSTM_AE_Wildfire":   AutoencoderLSTMModel(candidate_features=WILDFIRE_COLUMNS),
        "10_Transformer_Wildfire": VanillaTransformerModel(
            candidate_features=WILDFIRE_COLUMNS,
            seq_len=TRANSFORMER_SEQ_LEN,
        ),
    }

    for model_name, model in models_to_run.items():
        print(f"  -> Training {model_name}  [dataset={DATASET_MODE}]...")

        if isinstance(model, VanillaTransformerModel) and aug_data is not None:
            zone_key = region_name.replace(" ", "_")
            if f"{zone_key}_X_real" not in aug_data:
                print(f"    Warning: no augmented data for zone '{region_name}', skipping.")
                continue
            X_real = aug_data[f"{zone_key}_X_real"]
            y_real = aug_data[f"{zone_key}_y_real"]
            X_syn  = aug_data[f"{zone_key}_X_syn"]
            y_syn  = aug_data[f"{zone_key}_y_syn"]
            X_aug_arr = np.concatenate([X_real, X_syn], axis=0)
            y_aug_arr = np.concatenate([y_real, y_syn], axis=0)
            model.fit_augmented(X_aug_arr, y_aug_arr, val_df=val_data)
        elif isinstance(model, VanillaTransformerModel):
            model.fit(train_data, val_df=val_data)
        else:
            model.fit(train_data)

        model.tune_threshold(val_data)

        test_preds     = model.predict(test_data)
        processed_test = model.preprocess(test_data)
        actuals = (processed_test[TARGET_COL] > EXCEEDANCE_THRESHOLD).astype(int)

        all_results.append({
            "Region_Name":           region_name,
            "Model_Type":            f"{model_name} [{DATASET_MODE}]",
            "Probability_Threshold": round(model.alert_probability_threshold, 2),
            "Recall":    float(recall_score   (actuals, test_preds, zero_division=0)),
            "Precision": float(precision_score(actuals, test_preds, zero_division=0)),
            "Accuracy":  float(accuracy_score (actuals, test_preds)),
            "F1":        float(f1_score       (actuals, test_preds, zero_division=0)),
        })

        # 視覺化
        fig, ax = plt.subplots(figsize=(15, 5))
        test_dates  = processed_test.index
        actual_pm25 = processed_test[TARGET_COL].values

        ax.plot(test_dates, actual_pm25,
                label='Actual PM2.5', color='steelblue', alpha=0.7, linewidth=1.5)
        ax.axhline(EXCEEDANCE_THRESHOLD, color='gray', linestyle='--', linewidth=2,
                   label=f'Threshold ({EXCEEDANCE_THRESHOLD})')

        predicted_alert_indices = np.where(test_preds == 1)[0]
        ax.scatter(
            test_dates[predicted_alert_indices], actual_pm25[predicted_alert_indices],
            color='red', s=30, zorder=5, label='Model Warning (Predicted > 15 in 3 Hours)'
        )

        ax.set_title(f'[{region_name}] {model_name} [{DATASET_MODE}] — Test 2025',
                     fontsize=14, fontweight='bold')
        ax.set_xlabel('Date (UTC)', fontsize=12)
        ax.set_ylabel('PM2.5 Concentration', fontsize=12)
        ax.legend(loc='upper right')
        ax.grid(True, linestyle=':', alpha=0.6)
        plt.tight_layout()
        ipy_display(fig)
        plt.close(fig)

print("\n=== Final Classification Report ===")
results_df = pd.DataFrame(all_results)
if not results_df.empty:
    results_df = results_df.set_index(["Region_Name", "Model_Type"]).sort_index()
    try:
        ipy_display(results_df)
    except Exception:
        print(results_df)
else:
    print("No results generated.")

--- Running Unified Pipeline  [Mode=pchip]  Horizon=3 ---

Processing Region: Lower Fraser Valley ...
  -> Training 10_Transformer_Wildfire  [dataset=pchip]...
      [Tuning] Using User-Defined Validation Set (2024) for Optuna...


In [3]:
# 保存结果到 CSV（每跑完一个 DATASET_MODE 就运行这个 cell）
output_path = PROJECT_DIRECTORY / "Dataset" / "Outputs" / f"CS2_transformer_results_{DATASET_MODE}.csv"
results_df.reset_index().to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: D:\jason\UWaterloo\Term2\Stat 946\CS1\Data-Science-Case-Study-1\Dataset\Outputs\CS2_transformer_results_original.csv
